In [1]:
import os
import numpy as np
import h5py
from tqdm import tqdm

import matplotlib.pyplot as plt

In [2]:
def generate_initial_conditions(xc, n_samples, rng, k_tot=4, n_modes=2):
    """Generate random sinusoidal initial conditions, normalized to [0, 1].
    Mimics the init_multi function from the original codebase."""
    nx = len(xc)
    x_len = xc[-1] - xc[0]
    u_all = np.zeros((n_samples, nx), dtype=np.float32)

    for i in range(n_samples):
        # Select random wavenumber modes
        selected_k = rng.choice(k_tot, size=n_modes, replace=False) + 1
        u = np.zeros(nx, dtype=np.float64)
        for k in selected_k:
            amp = rng.uniform(0, 1)
            phase = rng.uniform(0, 2 * np.pi)
            wavenumber = 2 * np.pi * k / x_len
            u += amp * np.sin(wavenumber * xc + phase)
        # Random sign flip
        if rng.random() < 0.5:
            u = -u
        # Remove linear trend for periodicity
        slope = (u[-1] - u[0]) / x_len
        u -= slope * xc
        # Normalize to [0, 1]
        u_min, u_max = u.min(), u.max()
        if u_max - u_min > 1e-10:
            u = (u - u_min) / (u_max - u_min)
        u_all[i] = u.astype(np.float32)
    return u_all

In [3]:
from scipy.spatial.distance import cdist

def covariance(x,y, k_sigma, l):
    sq_norm = cdist(x, y, metric='sqeuclidean')
    return np.exp(-0.5 * sq_norm / l**2) * k_sigma**2
    

def Gaussian_Process(xc, nv, k_sigma, l):
    
    nx = len(xc)
    # generate parametric function
    K = covariance(xc.reshape(-1,1),xc.reshape(-1,1),k_sigma, l)
    alpha = np.random.multivariate_normal(mean=np.zeros(nx), cov=K)
    
    # periodic
    slope = (alpha[-1] - alpha[0]) / (xc[-1] - xc[0])
    alpha = alpha - np.outer(slope,xc).reshape(-1)
    
    # 
    min_val = np.min(alpha)
    max_val = np.max(alpha)
    
    alpha = (alpha - min_val) / (max_val - min_val)

    return nv * alpha

In [4]:
def solve_parametric_wave(u0, alpha_t, x_range, t_range, nx, t_num, CFL=0.4):
    """
    Solve:
        u_tt = alpha(t)^2 u_xx
    with periodic BC and u_t(0,x)=0
    """

    dx = x_range / nx
    dt_save = t_range / (t_num - 1)

    # periodic Laplacian
    def lap(u):
        return (np.roll(u, -1) - 2*u + np.roll(u, 1)) / dx**2

    # initial state
    u = u0.copy()
    v = np.zeros_like(u0)

    uu = np.zeros((t_num, nx), dtype=np.float32)
    uu[0] = u

    t = 0.0
    i_save = 1
    t_save = dt_save

    while i_save < t_num:

        a = alpha_t(t) if callable(alpha_t) else alpha_t[i_save - 1]

        # CFL condition
        dt = min(CFL * dx / (abs(a) + 1e-12), t_save - t)

        # leapfrog
        v += 0.5 * dt * (a**2) * lap(u)
        u += dt * v
        v += 0.5 * dt * (a**2) * lap(u)

        t += dt

        if t >= t_save - 1e-12:
            uu[i_save] = u.astype(np.float32)
            i_save += 1
            t_save = i_save * dt_save

    return uu

In [5]:
def generate_wave_dataset(data_dir, n_param_sets, ics_per_param, k_sigma, nv, l,
                          x_range, t_range, nx, t_num, file_name, seed=42):
    
    
    os.makedirs(data_dir, exist_ok=True)
    h5_path = os.path.join(data_dir, file_name)
    
    rng = np.random.default_rng(seed)
    dx = x_range / nx
    dt = t_range / t_num
    xc = np.linspace(0, x_range, nx + 1)[:-1] + 0.5 * dx
    tc = np.linspace(0, t_range, t_num+1)[:-1] + 0.5 * dt
    
    total = n_param_sets * ics_per_param
    all_solutions = np.zeros((total, t_num, nx, 1), dtype=np.float32)
    all_coeffs = np.zeros((total, t_num), dtype=np.float32)
    idx = 0
    failed = 0

    # Generate ICs for this parameter set
    ics = generate_initial_conditions(xc, ics_per_param * 2, rng)  # generate extra in case some fail
    
    for p in tqdm(range(n_param_sets), desc="Generating data"):
        # coefficient function from Gaussian Process
        coeff = Gaussian_Process(tc, nv, k_sigma, l) 

        count = 0
        for ic_idx in range(len(ics)):
            if count >= ics_per_param:
                break
            try:
                sol = solve_parametric_wave(ics[ic_idx].astype(np.float64), coeff, 
                                      x_range, t_range, nx, t_num, CFL=0.4)
                
                norm = np.linalg.norm(sol)
                if norm < 2000 and norm > 1 and np.all(np.isfinite(sol)):
                    all_solutions[idx, :, :, 0] = sol
                    all_coeffs[idx] = coeff
                    idx += 1
                    count += 1
                else:
                    failed += 1
            except Exception:
                failed += 1


    # Trim to actual size
    all_solutions = all_solutions[:idx]
    all_coeffs = all_coeffs[:idx]

    # Save
    with h5py.File(h5_path, "w") as hf:
        hf.create_dataset("data", data=all_solutions)
        hf.create_dataset("coeffs", data=all_coeffs)

    print(f"Saved {idx} samples to {h5_path}")
    return h5_path

In [6]:
# ============ Data / PDE config ============
DATA_DIR        = "dataset_simple/"   # where to save/load generated data
T_NUM           = 64                  # timesteps
X_NUM           = 128                 # spatial grid points
T_RANGE         = (0.0, 2.0)
X_RANGE         = (0.0, 2.0)
INPUT_LEN       = 1                   # input timesteps
INPUT_STEP      = 1
OUTPUT_STEP     = 2
NORMALIZE       = True

# Generation parameters
N_PARAM_SETS    = 400                 # number of different PDE coefficient sets
ICS_PER_PARAM   = 20                  # initial conditions per coefficient set
TOTAL_SAMPLES   = N_PARAM_SETS * ICS_PER_PARAM  # 2000 total

# PDE coefficient ranges
k_sigma = 1
nv = 1
l = 1

In [7]:
# Generate or load data
file_name = 'solutions.h5'
h5_path = os.path.join(DATA_DIR, file_name)

if os.path.exists(h5_path):
    print(f"Data already exists at {h5_path}, loading...")
    with h5py.File(h5_path, "r") as hf:
        print(f"  Solutions shape: {hf['data'].shape}")
        print(f"  Coefficients shape: {hf['coeffs'].shape}")
else:
    print("Generating conservation law dataset...")
    generate_wave_dataset(DATA_DIR, N_PARAM_SETS, ICS_PER_PARAM, k_sigma, nv, l,
                        X_RANGE[1], T_RANGE[1], X_NUM, T_NUM, file_name, seed=42)

Generating conservation law dataset...


Generating data: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 400/400 [00:43<00:00,  9.15it/s]


Saved 8000 samples to dataset_simple/solutions.h5
